# 厨二病ベクトルの作成（GLuCoSE-base-ja-v2）

## 1. 文埋め込み（BERT）

In [1]:
!pip -q install -U sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 15.5 MB/s eta 0:00:00


In [2]:
from pathlib import Path

def load_lines(path):
    lines = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                lines.append(s)
    return lines

chuni = load_lines("/content/chuni_sentences.txt")
normal = load_lines("/content/normal_sentences.txt")

print("厨二病文数:", len(chuni))
print("普通文数:", len(normal))
print("厨二病サンプル:", chuni[0])
print("普通文サンプル:", normal[0])

厨二病文数: 130
普通文数: 130
厨二病サンプル: 妖精さんたち！私に力を貸して！！！魅惑の香りに飲み込まれなさい"ポイズン・ベリー"！！！
普通文サンプル: コノリジン(Conolidine)は、アルカロイドの1つである


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 日本語でも比較的安定して使える多言語SBERT
MODEL_NAME = "pkshatech/GLuCoSE-base-ja-v2"
model = SentenceTransformer(MODEL_NAME)

E_chu = model.encode(chuni, normalize_embeddings=True, batch_size=32, show_progress_bar=True)
E_nor = model.encode(normal, normalize_embeddings=True, batch_size=32, show_progress_bar=True)

mu_chu = E_chu.mean(axis=0)
mu_nor = E_nor.mean(axis=0)

v_chu = mu_chu - mu_nor
v_chu = v_chu / np.linalg.norm(v_chu)  # 正規化（コサイン用）

np.save("chuni_vector.npy", v_chu)

print("v_chu shape:", v_chu.shape)
print("chuni_vector.npy を保存しました")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

v_chu shape: (768,)
chuni_vector.npy を保存しました


## 2. 厨二病ベクトルの定義

In [4]:
import numpy as np

# ベクトル読み込み
v_chu = np.load("chuni_vector.npy")

# すでに normalize_embeddings=True なので内積＝コサイン類似度
scores_chu = E_chu @ v_chu
scores_nor = E_nor @ v_chu

topk = 10

print("\n=== 厨二病文：上位10（厨二病度が高い） ===")
idx = np.argsort(-scores_chu)[:topk]
for i in idx:
    print(f"{scores_chu[i]:+.3f}  {chuni[i]}")

print("\n=== 普通文：下位10（厨二病度が低い） ===")
idx2 = np.argsort(scores_nor)[:topk]
for i in idx2:
    print(f"{scores_nor[i]:+.3f}  {normal[i]}")


=== 厨二病文：上位10（厨二病度が高い） ===
+0.459  俺の心が叫んでいるんだ！最高のエンディングをむかえろと！立て！まだ君はチャレンジしていないんだ！
+0.444  この戦いが終わったら...いやなんでもない。言わないよ。俺にも譲れないものがあるからな。
+0.444  あなたこんな力を持っていながら、何もできないの？そうだわ！私の仲間になるといいわ！
+0.439  心がざわめく。俺は、本当に応援してよかったのだろうか。勝ちを譲る...と言えば聞こえはいいが、挑戦から逃げただけだったのかもしれない。
+0.435  残念だな貴様では私には勝てないだろう！！なぜだと？！それは貴様が弱いからだ！弱者よ私に負けを認めて降参せよ。
+0.432  我が魂を代償に…捧げ…お前を召喚する。我がもとに来い！！兄弟なる英雄よ！！！....くそぉぉぉ！！なんでこねーんだよぉぉぉ！
+0.431  はぁはぁ、行くな！行ってくれるな！！俺を運ぶ鉄の塊、頼む待ってくれっ！！はぁはぁ次遅れたら怒ら..神の逆鱗に触れてしまうんだ！あ...あぁ終わった。
+0.426  これも運命か…俺にはかなり堪えたぞ。ふふふ…み、認めてやろう。お前の能力『未読スルー』を…
+0.425  君がいない世界なんて、色を失った幻想だ。俺はこれからどうしたらいいんだ...なぁ答えてくれ...よ
+0.422  フハハハハハ、ついに手に入れたぞ！！！俺はついに手に入れた！！これでだれも俺を止めることはできない！！フハハハハハ

=== 普通文：下位10（厨二病度が低い） ===
-0.184  地域学部（ちいきがくぶ）とは、人文科学・社会科学・自然科学を複合的に学び、日本国内の地域社会の創造を教育研究するために大学に置かれる学部の名称
-0.165  松本 泰丈（まつもと ひろたけ、1941年 - ）は、日本の言語学者・日本語学者
-0.158  牟礼 慶子（むれ けいこ、1929年〈昭和4年〉2月1日 - 2012年〈平成24年〉1月29日）は、日本の詩人・教師
-0.154  連語論、奄美方言研究で知られる
-0.143  また農民の顔を描いた習作もあり、そのうちの一つが広島のウッドワン美術館に収蔵されている
-0.142  京セラコネクタプロダクツ株式会社（きょうセラコネクタプロダクツ、K

## 3. 厨二病ベクトルの評価

In [5]:
print("平均スコア")
print("厨二病:", float(scores_chu.mean()))
print("普通  :", float(scores_nor.mean()))

平均スコア
厨二病: 0.33790823817253113
普通  : -0.07561712712049484
